## Compare the five models: speed vs precision

Reads the `benchmark_summary.csv` written by the benchmark section of each notebook (`<OUTPUT_DIR>-benchmark/benchmark_summary.csv`).
Upload the five files (or the five folders) next to this notebook and run the cells. Missing models are skipped.

### 1. Load the five summaries

In [ ]:
import glob

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

files = sorted(glob.glob("*-benchmark/benchmark_summary.csv") + glob.glob("*benchmark_summary*.csv"))
summary = pd.concat([pd.read_csv(f) for f in files], ignore_index=True).drop_duplicates()
print("Files:", files)
print("Models:", summary["model"].unique().tolist())

### 2. Ranking tables (one for speed, one for precision)

Rank 1 = best. For speed, lower is better for TTFT / total time / TTLT and higher is better for tokens per second.

In [ ]:
CONDITION = "fine-tuned"  # change to "base" to rank the base models

data = summary[summary["condition"] == CONDITION]
for category in ["speed", "precision"]:
    table = data[data["category"] == category].pivot_table(index="model", columns="metric", values="mean")
    better = data[data["category"] == category].groupby("metric")["better"].first()
    ranks = pd.DataFrame({m: table[m].rank(ascending=(better[m] == "lower")) for m in table.columns})
    table[f"{category} rank (mean)"] = ranks.mean(axis=1)
    print(f"\n{category.upper()} ({CONDITION})")
    display(table.sort_values(f"{category} rank (mean)").round(3))

### 3. Speed vs precision map (up and right is better)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for condition, marker in [("base", "o"), ("fine-tuned", "s")]:
    rows = summary[summary["condition"] == condition]
    speed = rows[rows["metric"] == "Tokens per second"].set_index("model")["mean"]
    precision = rows[rows["category"] == "precision"].groupby("model")["mean"].mean()
    ax.scatter(speed, precision.loc[speed.index], marker=marker, s=80, label=condition)
    for model in speed.index:
        ax.annotate(model, (speed[model], precision[model]), textcoords="offset points", xytext=(5, 5), fontsize=8)
ax.set_xlabel("tokens per second")
ax.set_ylabel("mean accuracy (MedQA, PubMedQA)")
ax.set_title("Speed vs precision")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()